In [ ]:
import os, glob, csv, math
import numpy as np
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device="cpu"

base="/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data"
test_dir=os.path.join(base,"test_images_v2","test_set_v2")

IMG=128

def load_img(p):
    i=cv2.imread(p,cv2.IMREAD_GRAYSCALE)
    if i is None: return None
    return cv2.resize(i,(IMG,IMG))

def extract(mask):
    l=cv2.HoughLinesP(mask,1,np.pi/180,30,minLineLength=20,maxLineGap=10)
    if l is None: return []
    return l[:,0,:]

def ang(l):
    return np.degrees(np.arctan2(l[3]-l[1],l[2]-l[0]))

def compute(mask):
    apo=extract((mask==2).astype(np.uint8)*255)
    fas=extract((mask==1).astype(np.uint8)*255)
    if len(apo)<2 or len(fas)<1:
        return 20,100,30
    mt=abs(apo[0][1]-apo[1][1])
    pas=[]
    for f in fas[:3]:
        a=abs(ang(f)-ang(apo[1]))
        if a>90:a=180-a
        pas.append(a)
    pa=np.median(pas)
    fl=mt/(math.sin(math.radians(pa))+1e-6)
    return float(np.clip(pa,5,45)),float(np.clip(fl,30,200)),float(np.clip(mt,10,50))

class SmallNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,16,3,2,1),nn.ReLU(),
            nn.Conv2d(16,32,3,2,1),nn.ReLU(),
            nn.Conv2d(32,64,3,2,1),nn.ReLU(),
            nn.ConvTranspose2d(64,32,2,2),nn.ReLU(),
            nn.ConvTranspose2d(32,16,2,2),nn.ReLU(),
            nn.ConvTranspose2d(16,3,2,2)
        )
    def forward(self,x): return self.net(x)

class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,8,3,2,1),nn.ReLU(),
            nn.Conv2d(8,16,3,2,1),nn.ReLU(),
            nn.ConvTranspose2d(16,8,2,2),nn.ReLU(),
            nn.ConvTranspose2d(8,3,2,2)
        )
    def forward(self,x): return self.net(x)

modelA=SmallNet().to(device)
modelB=TinyNet().to(device)

def predict_model(model,img):
    x=torch.tensor(img.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        seg=model(x)
        seg=nn.functional.interpolate(seg,size=(IMG,IMG),mode="bilinear",align_corners=False)
        seg=seg.argmax(1).squeeze().numpy()
    return compute(seg)

def ensemble(img):
    p1=predict_model(modelA,img)
    p2=predict_model(modelB,img)
    pa=0.6*p1[0]+0.4*p2[0]
    fl=0.6*p1[1]+0.4*p2[1]
    mt=0.6*p1[2]+0.4*p2[2]
    return pa,fl,mt

def tta(img):
    p1=ensemble(img)
    p2=ensemble(np.fliplr(img))
    return [(p1[i]+p2[i])/2 for i in range(3)]

test=sorted(glob.glob(os.path.join(test_dir,"*.png"))+glob.glob(os.path.join(test_dir,"*.tif")))

raw_preds=[]

for p in tqdm(test):
    img=load_img(p)
    if img is None:
        raw_preds.append([20,100,30])
    else:
        raw_preds.append(list(tta(img)))

raw_preds=np.array(raw_preds)

smoothed=np.copy(raw_preds)

for i in range(len(raw_preds)):
    l=max(0,i-2)
    r=min(len(raw_preds),i+3)
    smoothed[i]=raw_preds[l:r].mean(axis=0)

with open("submission.csv","w",newline="") as f:
    w=csv.writer(f)
    w.writerow(["image_id","pa_deg","fl_mm","mt_mm"])
    for i,p in enumerate(test):
        name=os.path.basename(p)
        pa,fl,mt=smoothed[i]
        w.writerow([name,pa,fl,mt])